In [ ]:
from pathlib import Path
from collections import defaultdict
import json
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from collections import Counter

import numpy as np
import mat73
from PIL import Image
from tqdm.auto import tqdm
import h5py
import torch
import torchvision.transforms as T


# THINGS

In [ ]:
things_image_database_dir = '${MBS_THINGS_RAW_DIR}/image_db/images/object_images'

things_image_database_dir = Path(things_image_database_dir)

assert things_image_database_dir.exists()

In [ ]:
things_image_database_dir

In [ ]:
# # Load and transform the training images

# stimuli_imgs = []

# num_workers = 8

# def process_image(img_path):
#     img = Image.open(img_path).convert('RGB')
#     # img = img.resize((256, 256), Image.BICUBIC)
#     img = np.array(img)
#     return img

# # with ProcessPoolExecutor(max_workers=num_workers) as executor:
# with ThreadPoolExecutor(max_workers=num_workers) as executor:
#     futures = []
#     for concep_folder in tqdm(list(things_image_database_dir.iterdir())):
#         for img_file in concep_folder.glob('*.jpg'):
#             img_path = img_file
#             futures.append(executor.submit(process_image, img_path))
    
#     for future in tqdm(futures, total=len(futures)):
#         img = future.result()
#         stimuli_imgs.append(img)

In [ ]:
# Counter([img.shape for img in stimuli_imgs])

In [ ]:
class THINGSImageDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        root_dir: str | Path,
        transform=None,
        target_transform=None,
    ):
        self.root_dir = Path(root_dir)
        self.image_paths = sorted(list(self.root_dir.glob('*/*.jpg')))
        self.transform = transform
        self.target_transform = target_transform
        self.default_transform = T.Compose([
            T.Resize((256, 256)),
            T.ToTensor(),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert('RGB')
        
        if self.transform is not None:
            x = self.transform(img)
        else:
            x = self.default_transform(img)

        if self.target_transform is not None:
            x = self.target_transform(x)
            
        img_loc = str(img_path.relative_to(self.root_dir))

        return img_loc, x

In [ ]:
dataset = THINGSImageDataset(
    root_dir=things_image_database_dir,
)
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
)



In [ ]:
batch = next(iter(dataloader))
batch

# NSD

In [ ]:
nsd_stimuli_path = "${MBS_NSD_DIR}/nsd_stimuli.hdf5"
nsd_stimuli_path = Path(nsd_stimuli_path)
assert nsd_stimuli_path.exists()

In [ ]:
# with h5py.File(nsd_stimuli_path, 'r') as f:
#     print(f['imgBrick'])
#     nsd_stimuli_images = f['imgBrick'][:]
#     # nsd_stimuli_imgs = []
#     # for i in range(len(f['stimuli'])):
#     #     img = f['stimuli'][i]
#     #     nsd_stimuli_imgs.append(img)

In [ ]:
# nsd_stimuli_images.shape

In [ ]:
# del nsd_stimuli_images

In [ ]:
class H5ImageDataset(torch.utils.data.Dataset):
    """
    Memory-efficient dataset for images stored in a single HDF5 file.

    Expected HDF5 layout (example):

        /${x_key}  -> shape (N, H, W, C)
        /${y_key}  -> shape (N, ... )  - optional

    Only the requested index is read into memory on each __getitem__ call.
    """

    def __init__(
        self,
        h5_path: str,
        x_key: str = "imgBrick",
        y_key: str | None = None,
        transform=None,
        target_transform=None,
    ):
        self.h5_path = h5_path
        self.x_key = x_key
        self.y_key = y_key
        self.transform = transform
        self.target_transform = target_transform
        self.default_transform = T.Compose([
            T.Resize((256, 256)),
            T.ToTensor(),
        ])

        # We only open the file briefly here to get the length.
        # The long-lived handle will be opened lazily in each process.
        with h5py.File(self.h5_path, "r") as f:
            self._length = f[self.x_key].shape[0]

        # Long-lived handles (per process), opened on first access
        self._file = None
        self._x = None
        self._y = None

    # ----------------- internal helpers -----------------

    def _ensure_file_open(self):
        """Open HDF5 file and datasets on first use in this process."""
        if self._file is None:
            # read-only is enough; one handle per process/worker
            self._file = h5py.File(self.h5_path, "r")
            self._x = self._file[self.x_key]
            if self.y_key is not None:
                self._y = self._file[self.y_key]

    def _close_file(self):
        """Close HDF5 file if it's open."""
        if self._file is not None:
            try:
                self._file.close()
            except Exception:
                pass
            self._file = None
            self._x = None
            self._y = None

    # ----------------- PyTorch Dataset API -----------------

    def __len__(self):
        return self._length

    def __getitem__(self, idx):
        self._ensure_file_open()

        # HDF5 slice -> NumPy array
        x = self._x[idx]

        # Convert to PIL.Image
        x = Image.fromarray(x)

        if self.transform is not None:
            x = self.transform(x)
        else:
            x = self.default_transform(x)

        if self._y is not None:
            y = self._y[idx]
            y = torch.from_numpy(y)
            if self.target_transform is not None:
                y = self.target_transform(y)
            return idx, x, y

        return idx, x

    # Make sure file handle is closed when dataset is garbage-collected
    def __del__(self):
        self._close_file()

In [ ]:
dataset = H5ImageDataset(nsd_stimuli_path)

dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)

In [ ]:
batch = next(iter(dataloader))
# batch[1][0].shape
batch

# Brain-Score

In [ ]:
from brainscore_vision import load_dataset, load_stimulus_set, load_benchmark
import pandas as pd

In [ ]:
class BrainScoreStimulusDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        stimulus_set_id: str,
        transform=None,
    ):
        self.stimulus_set_id = stimulus_set_id
        self.stimulus_set = load_stimulus_set(stimulus_set_id)
        self.stimulus_set = self.stimulus_set.sort_values('stimulus_id').reset_index(drop=True)
        self.transform = transform
        self.default_transform = T.Compose([
            T.Resize((256, 256)),
            T.ToTensor(),
        ])

    def __len__(self):
        return len(self.stimulus_set)

    def __getitem__(self, idx):
        stimulus = self.stimulus_set.iloc[idx]
        stimulus_id = stimulus.stimulus_id
        img_path = self.stimulus_set.get_stimulus(stimulus_id)
        
        img = Image.open(img_path).convert('RGB')

        if self.transform is not None:
            x = self.transform(img)
        else:
            x = self.default_transform(img)
            

        return stimulus_id, x

### FreemanZiemba2013

In [ ]:
# # benchmark = load_benchmark('FreemanZiemba2013public.V1-pls')
# # data = load_dataset('FreemanZiemba2013.public')
# stimuli_set =  load_stimulus_set('FreemanZiemba2013.aperture-public')

In [ ]:
# stimuli_set.get_stimulus("21041db1f26c142812a66277c2957fb3e2070916")

In [ ]:
dataset = BrainScoreStimulusDataset(
    stimulus_set_id='FreemanZiemba2013.aperture-public',
)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)

batch = next(iter(dataloader))
batch

### MajajHong2015

In [ ]:
# # benchmark = load_benchmark('MajajHong2015public.IT-pls')
# # data = load_dataset('MajajHong2015.public')
# stimuli_set =  load_stimulus_set('hvm-public')

In [ ]:
# data

In [ ]:
# stimuli_set

In [ ]:
dataset = BrainScoreStimulusDataset(
    stimulus_set_id='hvm-public'
)

dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)

batch = next(iter(dataloader))
batch

In [ ]:
dataset[0][0]